# 🛡️ FraudShield AI: Google Colab Training Pipeline

Train **GraphSAGE GNN** (Signup Trust Model) and **LightGBM** (Payment Abuse Model) directly from the 10M dataset hosted on Hugging Face (`vicky1428/fraudshield-10m`) using GPU acceleration.

---
### Models Trained:
1. **Signup Trust Model (`SignupGraphSAGE`)**: Predicts user trust score (0-100), risk score, and 4-tier signup action.
2. **Payment Abuse Model (`PaymentAbuseModel`)**: Multi-class LightGBM (`0=legit`, `1=trial_abuse`, `2=discount_abuse`, `3=payment_fraud`) with tiered decisioning (`ALLOW`, `ALLOW_FLAG_REVIEW`, `ALLOW_HIGH_PRIORITY_REVIEW`, `BLOCK`).

## 1. Check GPU Compute Device
Colab provides free T4 GPUs. Go to **Runtime > Change runtime type > T4 GPU** if not already enabled.

In [ ]:
!nvidia-smi

## 2. Clone Repository & Install Dependencies

In [ ]:
# 1. Clone repo if running in a fresh Colab environment
import os
if not os.path.exists("razorpay"):
    !git clone https://github.com/manideep1428/razorpay.git
    %cd razorpay
else:
    %cd razorpay

# 2. Install dependencies
!pip install -q torch torch-geometric lightgbm scikit-learn pandas numpy joblib datasets huggingface-hub pyarrow matplotlib

## 3. (Optional) Mount Google Drive to Persist Checkpoints
Uncomment the cell below if you want trained checkpoints saved directly to your Google Drive.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# SAVE_DIR = '/content/drive/MyDrive/fraudshield_models'

SAVE_DIR = 'artifacts'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Models will be saved to: {SAVE_DIR}")

## 4. Configure Training Parameters

Choose your training scale:
* **Fast verification**: `MAX_ROWS = 50_000` (~1 minute)
* **Standard benchmark**: `MAX_ROWS = 500_000` (~4 minutes on T4 GPU)
* **Large scale**: `MAX_ROWS = 2_000_000` (~15 minutes on T4 GPU)
* **Full 10M dataset**: Set `MAX_ROWS = None`

In [ ]:
REPO_ID = "vicky1428/fraudshield-10m"
MAX_ROWS = 500_000        # Adjust as desired (e.g., 50000, 500000, 2000000, or None)
GNN_EPOCHS = 100          # GraphSAGE epochs
LIGHTGBM_TREES = 300      # LightGBM boosting trees
DEVICE = "cuda"           # Uses NVIDIA GPU
CALIBRATE = True          # Enable probability calibration

max_rows_flag = f"--max-rows {MAX_ROWS}" if MAX_ROWS else ""
calib_flag = "--calibrate" if CALIBRATE else ""

cmd = f"python train.py --repo-id {REPO_ID} {max_rows_flag} --epochs {GNN_EPOCHS} --trees {LIGHTGBM_TREES} --device {DEVICE} --save-dir {SAVE_DIR} {calib_flag}"
print("Training command:", cmd)

## 5. Execute Training on Hugging Face Dataset

In [ ]:
!{cmd}

## 6. Evaluate Models on Held-Out Test Split from Hugging Face

In [ ]:
!python test.py --repo-id {REPO_ID} --test-rows 20000 --artifacts-dir {SAVE_DIR}

## 7. Interactive Live Inference Demo
Test individual signup decisions and payment transactions directly with the trained predictors.

In [ ]:
import sys
sys.path.insert(0, 'src')
import pandas as pd
from trust_radar.inference.predict_signup import SignupPredictor
from trust_radar.inference.predict_payment import PaymentPredictor
from trust_radar.config import FeatureConfig
from trust_radar.utils.synthetic import synthesize_payment_dataset

# 1. Test Payment Predictor
pay_predictor = PaymentPredictor(f"{SAVE_DIR}/payment_abuse_lgbm.joblib")
sample_txs = synthesize_payment_dataset(n=5, seed=123)
results = pay_predictor.score_batch(sample_txs)

display_cols = ['payment_risk_score', 'abuse_type', 'risk_level', 'decision']
print("--- Sample Payment Transactions Scoring ---")
print(results[display_cols].to_string(index=False))

# 2. Test Signup Predictor Checkpoint
signup_predictor = SignupPredictor(f"{SAVE_DIR}/signup_graphsage.pt")
print("\n[SUCCESS] Both model predictors loaded and ready for production inference!")

## 8. (Optional) Push Trained Models to Hugging Face Hub
Save your trained model checkpoints into your Hugging Face model repository.

In [ ]:
# from huggingface_hub import HfApi, create_repo
# HF_MODEL_REPO = "your-username/fraudshield-models"
# HF_TOKEN = "hf_..."
# api = HfApi(token=HF_TOKEN)
# create_repo(HF_MODEL_REPO, repo_type="model", token=HF_TOKEN, exist_ok=True)
# api.upload_file(path_or_fileobj=f"{SAVE_DIR}/signup_graphsage.pt", path_in_repo="signup_graphsage.pt", repo_id=HF_MODEL_REPO, repo_type="model")
# api.upload_file(path_or_fileobj=f"{SAVE_DIR}/payment_abuse_lgbm.joblib", path_in_repo="payment_abuse_lgbm.joblib", repo_id=HF_MODEL_REPO, repo_type="model")
# print(f"Trained models uploaded to https://huggingface.co/{HF_MODEL_REPO}")